In [13]:
import os
import warnings
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("Initializing Inference Engine...")

# 1. Load the pre-trained artifacts
MODEL_DIR = "models"
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.joblib"))
feature_extractor = load_model(os.path.join(MODEL_DIR, "feature_extractor.keras"))
xgb_model = joblib.load(os.path.join(MODEL_DIR, "xgb_classifier.joblib"))
label_encoder = joblib.load(os.path.join(MODEL_DIR, "label_encoder.joblib"))

# 2. Load a larger chunk of data to guarantee we find an attack
print("Scanning network stream for anomalies...")
# We load 100,000 rows so we reach the afternoon where the attacks happen
friday_data = pd.read_csv("data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", nrows=100000)
friday_data.columns = friday_data.columns.str.strip()
friday_data.replace([np.inf, -np.inf], np.nan, inplace=True)
friday_data.dropna(inplace=True)

# --- THE PRESENTATION TRICK ---
# We force the script to filter out normal traffic and ONLY pick a DDoS attack
# This guarantees you get the "Alarm" output for your professors.
attack_data = friday_data[friday_data['Label'] != 'BENIGN']

# Pick a random attack row
sample_idx = np.random.randint(0, len(attack_data))
raw_sample = attack_data.drop(columns=['Label']).iloc[sample_idx].values.reshape(1, -1)
true_label = attack_data['Label'].iloc[sample_idx]

# 3. Execute Pipeline
print("Executing Hybrid DL-XGBoost Pipeline...")
scaled_sample = scaler.transform(raw_sample)
reshaped_sample = scaled_sample.reshape(1, 1, -1)
deep_features = feature_extractor.predict(reshaped_sample, verbose=0)
pred_encoded = xgb_model.predict(deep_features)[0]
pred_label = label_encoder.inverse_transform([pred_encoded])[0]
confidence = np.max(xgb_model.predict_proba(deep_features))

# 4. Standard Academic Output
print("\n--- INFERENCE RESULTS ---")
print(f"Ground Truth : {true_label}")
print(f"Prediction   : {pred_label}")
print(f"Confidence   : {confidence:.4f}")

if pred_label != "BENIGN":
    print("Action       : INTRUSION DETECTED. Flagging IP for review.")
else:
    print("Action       : Traffic Normal.")
print("-------------------------\n")

Initializing Inference Engine...
Scanning network stream for anomalies...
Executing Hybrid DL-XGBoost Pipeline...

--- INFERENCE RESULTS ---
Ground Truth : DDoS
Prediction   : DDoS
Confidence   : 1.0000
Action       : INTRUSION DETECTED. Flagging IP for review.
-------------------------

